# Soft Frame Anchors in FRUST TS Guesses

Soft frame anchors are orientation handles. They are not optimizer constraints.

This notebook shows why the screen TS guess builder uses them for TS3 and TS4. A single hard role atom such as `substrate_C` or `pin_B` fixes where a fragment should sit, but it does not fix how the aromatic substrate or HBpin fragment is rotated around that point. Nearby soft frame atoms give the rigid fragment placement enough local geometry to put the ring face and HBpin O-B-O frame in a chemically useful orientation.

The important distinction is:

| Anchor type | What it does during TS guess generation | What happens after placement |
| --- | --- | --- |
| Hard role atom | Defines the reactive TS core position, for example `cat_B`, `pin_B`, or `substrate_C` | Snapped exactly onto the template coordinate |
| Soft frame atom | Helps the least-squares rigid fit orient a whole fragment | Not snapped exactly; it can move after the hard anchors are restored |

## Build a small screen example

The example below uses the same public screen path users normally call: read components, expand them into substrate-catalyst systems, then create TS3 and TS4 guesses.

In [2]:
import frust as ft
import numpy as np
import pandas as pd

In [3]:
components = ft.screen.read(
    pd.DataFrame(
        {
            "role": ["substrate", "catalyst"],
            "smiles": ["C1=CC=CO1", "BC1=C(N(C)C)C=CC=C1"],
            "compound_name": ["furan", "dimethylaminophenyl_borane"],
            "rpos": [2, pd.NA],
        }
    )
)

systems = ft.screen.expand(components)
ts_guesses = ft.screen.create_ts_guesses(
    systems,
    ts_types=["TS3", "TS4"],
    n_confs=3,
)

ts3 = ts_guesses["TS3"].reset_index(drop=True)
ts4 = ts_guesses["TS4"].reset_index(drop=True)
systems[["system_name", "substrate_name", "catalyst_name", "rpos"]]

,system_name,substrate_name,catalyst_name,rpos
0,furan__dimethylaminophenyl_borane,furan,dimethylaminophenyl_borane,2


## The normal TS guess

This is the regular FRUST result. The view below is not a special diagnostic path: it is the same `ft.vis.ts_guess_scene(...)` viewer used for screen outputs.

The angle and distance guides are optimizer constraints from `constraint_spec`. Soft frame anchors are different: they are only used while orienting the disconnected fragments before the final TS guess dataframe is made.

In [4]:
ft.vis.show_scene(
    ft.vis.ts_guess_scene(
        ts3,
        row_indices=[0],
        show_roles=True,
        show_constraint_distances=True,
        show_constraint_angles=True,
        columns=1,
        cell_size=(760, 560),
    )
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
ft.vis.show_scene(
    ft.vis.ts_guess_scene(
        ts4,
        row_indices=[0],
        show_roles=True,
        show_constraint_distances=True,
        show_constraint_angles=True,
        columns=1,
        cell_size=(760, 560),
    )
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Hard roles and soft frame atoms

The TS builder stores named reactive atoms in `constraint_roles`. Those role atoms are the hard anchors. The two substrate frame atoms and two HBpin oxygen frame atoms are nearby atoms chosen from the assembled molecule. They are not named optimizer roles; they are temporary orientation handles used during rigid placement.

The helpers below inspect the generated row and build a visual overlay:

| Color | Meaning |
| --- | --- |
| Cyan | Hard TS role atom, snapped exactly after placement |
| Orange | Substrate soft frame atom, used to orient the substrate ring |
| Yellow | HBpin soft frame atom, used to orient the O-B-O frame |

In [6]:
from collections.abc import Mapping
from contextlib import contextmanager

from tooltoad.scene3d import (
    AtomHighlight,
    AtomLabel,
    DistanceOverlay,
    GridScene,
    MoleculeModel,
    SceneCell,
)

HARD_ROLE_NAMES = ("cat_B", "cat_N", "cat_H", "transfer_H", "pin_B", "substrate_C")
VDW_RADII = {"H": 1.20, "B": 1.92, "C": 1.70, "N": 1.55, "O": 1.52}

def _row_bonds(row):
    bonds = row.get("connectivity_bonds", [])
    if bonds is None:
        return []
    return [(int(a), int(b)) for a, b in bonds]

def _bond_neighbors(row, atom_idx, *, symbols=None, exclude=()):
    exclude = {int(idx) for idx in exclude if idx is not None}
    neighbors = []
    for a, b in _row_bonds(row):
        if a == int(atom_idx) and b not in exclude:
            neighbors.append(b)
        elif b == int(atom_idx) and a not in exclude:
            neighbors.append(a)
    if symbols is not None:
        allowed = set(symbols)
        neighbors = [idx for idx in neighbors if str(row["atoms"][idx]) in allowed]
    return neighbors

def anchor_indices(row):
    roles = row.get("constraint_roles", {})
    if not isinstance(roles, Mapping):
        return {}, [], []

    hard = {role: int(roles[role]) for role in HARD_ROLE_NAMES if role in roles}

    substrate_soft = []
    if "substrate_C" in roles:
        excluded = [roles.get("cat_B"), roles.get("pin_B")]
        substrate_soft = _bond_neighbors(row, roles["substrate_C"], exclude=excluded)
        substrate_soft = [idx for idx in substrate_soft if str(row["atoms"][idx]) != "H"][:2]

    hbpin_soft = []
    if "pin_B" in roles:
        hbpin_soft = _bond_neighbors(row, roles["pin_B"], symbols={"O"})[:2]

    return hard, substrate_soft, hbpin_soft

def anchor_cell(row, *, title, coords_col="coords_embedded"):
    hard, substrate_soft, hbpin_soft = anchor_indices(row)
    overlays = []

    for role, atom_idx in hard.items():
        overlays.extend(
            [
                AtomHighlight(atom=atom_idx, color="cyan", radius=0.54, alpha=0.35),
                AtomLabel(atom=atom_idx, text=f"hard: {role}"),
            ]
        )

    for number, atom_idx in enumerate(substrate_soft, start=1):
        overlays.extend(
            [
                AtomHighlight(atom=atom_idx, color="orange", radius=0.5, alpha=0.4),
                AtomLabel(atom=atom_idx, text=f"soft substrate {number}"),
            ]
        )

    for number, atom_idx in enumerate(hbpin_soft, start=1):
        overlays.extend(
            [
                AtomHighlight(atom=atom_idx, color="yellow", radius=0.5, alpha=0.45),
                AtomLabel(atom=atom_idx, text=f"soft HBpin O {number}"),
            ]
        )

    return SceneCell(
        title=title,
        models=[
            MoleculeModel(
                atoms=row["atoms"],
                coords=row[coords_col],
                bonds=_row_bonds(row),
                style={"stick": {}, "sphere": {"radius": 0.28}},
            )
        ],
        overlays=overlays,
    )

def anchor_scene(df, *, row_index=0, title=None, coords_col="coords_embedded"):
    row = df.iloc[int(row_index)]
    if title is None:
        title = str(row.get("ts_type", "TS guess"))
    return GridScene(
        cells=[anchor_cell(row, title=title, coords_col=coords_col)],
        columns=1,
        cell_size=(760, 560),
        linked=False,
    )

def comparison_scene(current_df, no_soft_df, *, ts_type, row_index=0):
    return GridScene(
        cells=[
            anchor_cell(
                current_df.iloc[int(row_index)],
                title=f"{ts_type}: current hard + soft frame anchors",
            ),
            anchor_cell(
                no_soft_df.iloc[int(row_index)],
                title=f"{ts_type}: hard anchors only",
            ),
        ],
        columns=2,
        cell_size=(560, 520),
        linked=True,
    )

def _local_graph_pairs(row, *, max_depth=3):
    atoms = list(row["atoms"])
    adjacency = [set() for _ in atoms]
    for a, b in _row_bonds(row):
        adjacency[a].add(b)
        adjacency[b].add(a)

    excluded = set()
    for start in range(len(atoms)):
        seen = {start}
        frontier = {start}
        for _ in range(max_depth):
            next_frontier = set()
            for atom_idx in frontier:
                next_frontier.update(adjacency[atom_idx])
            next_frontier -= seen
            excluded.update(tuple(sorted((start, atom_idx))) for atom_idx in next_frontier)
            seen.update(next_frontier)
            frontier = next_frontier
    return excluded

def close_contact_records(row, *, cutoff=0.58, limit=None):
    atoms = list(row["atoms"])
    coords = np.asarray(row["coords_embedded"], dtype=float)
    excluded = _local_graph_pairs(row, max_depth=3)

    roles = row.get("constraint_roles", {}) or {}
    for constraint in row.get("constraint_spec", []) or []:
        if not isinstance(constraint, dict) or constraint.get("kind") != "distance":
            continue
        constraint_roles = constraint.get("roles") or []
        if len(constraint_roles) < 2:
            continue
        left, right = constraint_roles[:2]
        if left in roles and right in roles:
            excluded.add(tuple(sorted((int(roles[left]), int(roles[right])))))

    contacts = []
    for i, atom_i in enumerate(atoms):
        for j, atom_j in enumerate(atoms[i + 1 :], start=i + 1):
            if (i, j) in excluded:
                continue
            distance = float(np.linalg.norm(coords[i] - coords[j]))
            radius_sum = VDW_RADII.get(str(atom_i), 1.70) + VDW_RADII.get(str(atom_j), 1.70)
            normalized = distance / radius_sum
            if normalized < cutoff:
                contacts.append(
                    {
                        "atom1": i,
                        "atom2": j,
                        "symbols": f"{atom_i}-{atom_j}",
                        "distance_A": distance,
                        "normalized_vdw": normalized,
                    }
                )

    contacts.sort(key=lambda record: record["normalized_vdw"])
    if limit is not None:
        contacts = contacts[: int(limit)]
    return contacts

def close_contact_score(row, *, cutoff=0.58):
    contacts = close_contact_records(row, cutoff=cutoff)
    score = sum((cutoff - record["normalized_vdw"]) ** 2 for record in contacts)
    return score, len(contacts)

def clash_cell(row, *, title, coords_col="coords_embedded", cutoff=0.58, max_pairs=12):
    cell = anchor_cell(row, title=title, coords_col=coords_col)
    overlays = list(cell.overlays or [])
    for record in close_contact_records(row, cutoff=cutoff, limit=max_pairs):
        overlays.append(
            DistanceOverlay(
                atom1=int(record["atom1"]),
                atom2=int(record["atom2"]),
                label=f"{record['distance_A']:.2f} A",
                color="red",
                radius=0.045,
            )
        )
    return SceneCell(title=cell.title, models=cell.models, overlays=overlays)

In [18]:
viewer = ft.vis.show_scene(anchor_scene(ts3, row_index=0, title="TS3 hard roles and soft frame atoms"))
viewer.write_html("/Users/jacobmolinnielsen/Developer/Misc/pages/001_FRUSTv2/TS1_softframeanchors.html")
viewer

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The cyan atoms define the named TS core and are the atoms FRUST snaps exactly. The orange and yellow atoms are only there to give the rigid fragment fitting step enough geometry to know which face or local plane should point into the TS core.

In [8]:
ft.vis.show_scene(anchor_scene(ts4, row_index=0, title="TS4 hard roles and soft frame atoms"))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Mental model: one point, one axis, one plane

Rigid placement uses the coordinates in the placement map as matching points. More points remove more rotational freedom:

| Placement map | What it fixes | What remains ambiguous |
| --- | --- | --- |
| One anchor | Position | Rotation around that point |
| Two anchors | Position plus an axis | Rotation around that axis |
| Three non-collinear anchors | Position, axis, and local plane | The least-squares fit has a defined local orientation |

For a substrate, `substrate_C` is the point and the two neighboring heavy atoms define the local ring frame. For HBpin, `pin_B` is the point and the two oxygen neighbors define the O-B-O frame.

In [9]:
def toy_anchor_cell(title, points, labels):
    atoms = ["C"] * len(points)
    overlays = []
    for idx, label in enumerate(labels):
        overlays.extend(
            [
                AtomHighlight(atom=idx, color="cyan", radius=0.5, alpha=0.35),
                AtomLabel(atom=idx, text=label),
            ]
        )
    return SceneCell(
        title=title,
        models=[
            MoleculeModel(
                atoms=atoms,
                coords=np.asarray(points, dtype=float),
                bonds=[(idx, idx + 1) for idx in range(len(points) - 1)],
                style={"stick": {"radius": 0.08}, "sphere": {"radius": 0.32}},
            )
        ],
        overlays=overlays,
    )

toy_scene = GridScene(
    cells=[
        toy_anchor_cell("1 anchor: position only", [[0, 0, 0]], ["point"]),
        toy_anchor_cell("2 anchors: local axis", [[-0.8, 0, 0], [0.8, 0, 0]], ["A", "B"]),
        toy_anchor_cell(
            "3 anchors: local plane",
            [[-0.8, 0, 0], [0.8, 0, 0], [0.0, 0.9, 0.25]],
            ["A", "B", "C"],
        ),
    ],
    columns=3,
    cell_size=(330, 300),
    linked=False,
)
ft.vis.show_scene(toy_scene)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## What happens if the soft frame anchors are removed?

The code below is a local demonstration. It temporarily disables the substrate and HBpin frame coordinate maps, then regenerates the same TS guesses. This is not recommended production code; it is just a compact way to see what the soft frame atoms are doing.

With hard anchors only, the reactive atoms still land in roughly the right TS core positions. The problem is that the surrounding fragment orientation is underdetermined. Rotate the linked viewers and inspect the substrate ring face, HBpin O-B-O frame, and close contacts near the reactive center.

In [10]:
import frust.tsguess.assembly as ts_assembly

@contextmanager
def without_soft_frame_anchors():
    original_substrate_frames = ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS
    original_hbpin_frames = ts_assembly.HBPIN_FRAME_COORDS
    try:
        ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS = {}
        ts_assembly.HBPIN_FRAME_COORDS = {}
        yield
    finally:
        ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS = original_substrate_frames
        ts_assembly.HBPIN_FRAME_COORDS = original_hbpin_frames

with without_soft_frame_anchors():
    no_soft_ts_guesses = ft.screen.create_ts_guesses(
        systems,
        ts_types=["TS3", "TS4"],
        n_confs=3,
    )

no_soft_ts3 = no_soft_ts_guesses["TS3"].reset_index(drop=True)
no_soft_ts4 = no_soft_ts_guesses["TS4"].reset_index(drop=True)

In [11]:
ft.vis.show_scene(comparison_scene(ts3, no_soft_ts3, ts_type="TS3", row_index=0))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
ft.vis.show_scene(comparison_scene(ts4, no_soft_ts4, ts_type="TS4", row_index=0))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## A more catastrophic hard-only stress test

The furan example is intentionally small, so the hard-only result can look only mildly wrong. To make the failure mode obvious, use a bulkier substrate and catalyst, generate more hard-only conformers, and pick the hard-only TS3 row with the worst severe nonbonded contacts.

This is a diagnostic stress test, not a recommended workflow. The red guides below mark atom pairs that are much closer than their van der Waals radii would allow, after excluding ordinary bonded neighborhoods and the intended reactive distance constraints.

In [13]:
stress_components = ft.screen.read(
    pd.DataFrame(
        {
            "role": ["substrate", "catalyst"],
            "smiles": [
                "CC(C)(C)C1=CC=CC=C1",
                "BC1=C(N2CCCCC2)C=CC=C1",
            ],
            "compound_name": ["tertbutyl_benzene", "piperidine_cat"],
            "rpos": [5, pd.NA],
        }
    )
)
stress_systems = ft.screen.expand(stress_components)
stress_ts3 = ft.screen.create_ts_guesses(
    stress_systems,
    ts_types=["TS3"],
    n_confs=16,
)["TS3"].reset_index(drop=True)

with without_soft_frame_anchors():
    stress_no_soft_ts3 = ft.screen.create_ts_guesses(
        stress_systems,
        ts_types=["TS3"],
        n_confs=16,
    )["TS3"].reset_index(drop=True)

stress_systems[["system_name", "substrate_name", "catalyst_name", "rpos"]]

,system_name,substrate_name,catalyst_name,rpos
0,tertbutyl_benzene__piperidine_cat,tertbutyl_benzene,piperidine_cat,5


In [14]:
hard_only_scores = [
    (row_index, *close_contact_score(row))
    for row_index, row in stress_no_soft_ts3.iterrows()
]
stress_worst_no_soft_idx, stress_worst_score, stress_worst_n_contacts = max(
    hard_only_scores,
    key=lambda item: (item[1], item[2]),
)

current_contacts = close_contact_records(stress_ts3.iloc[0])
hard_only_contacts = close_contact_records(
    stress_no_soft_ts3.iloc[stress_worst_no_soft_idx]
)
stress_clash_summary = pd.DataFrame(
    [
        {
            "case": "current soft-frame TS3 row 0",
            "row": 0,
            "severe_contacts": len(current_contacts),
            "worst_distance_A": min(
                (record["distance_A"] for record in current_contacts),
                default=pd.NA,
            ),
        },
        {
            "case": "hard-only TS3 worst sampled row",
            "row": stress_worst_no_soft_idx,
            "severe_contacts": len(hard_only_contacts),
            "worst_distance_A": min(
                (record["distance_A"] for record in hard_only_contacts),
                default=pd.NA,
            ),
        },
    ]
)
stress_clash_summary

,case,row,severe_contacts,worst_distance_A
0,current soft-frame TS3 row 0,0,3,0.925951
1,hard-only TS3 worst sampled row,11,22,0.720856


In [15]:
pd.DataFrame(hard_only_contacts[:10])

,atom1,atom2,symbols,distance_A,normalized_vdw
0,27,63,H-H,0.720856,0.300357
1,27,33,H-C,1.108059,0.382089
2,27,34,H-C,1.144366,0.394609
3,34,63,C-H,1.246488,0.429823
4,33,63,C-H,1.274382,0.439442
5,12,34,C-C,1.615244,0.475072
6,40,51,H-C,1.405042,0.484497
7,0,43,B-H,1.537091,0.492657
8,35,63,C-H,1.431008,0.493451
9,33,54,C-O,1.596437,0.495788


In [16]:
stress_scene = GridScene(
    cells=[
        clash_cell(
            stress_ts3.iloc[0],
            title="Current soft-frame TS3, row 0",
            max_pairs=12,
        ),
        clash_cell(
            stress_no_soft_ts3.iloc[stress_worst_no_soft_idx],
            title=f"Hard-only TS3, worst sampled row {stress_worst_no_soft_idx}",
            max_pairs=12,
        ),
    ],
    columns=2,
    cell_size=(600, 560),
    linked=True,
)
ft.vis.show_scene(stress_scene)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## How this connects to the implementation

The current screen TS path separates identity from orientation:

1. `constraint_roles` gives named reactive atoms such as `cat_B`, `pin_B`, and `substrate_C`.
2. The placement coordinate map adds those hard role atoms plus nearby soft frame atoms for substrate and HBpin orientation.
3. Fragment placement uses a least-squares rigid transform from source anchors to template target coordinates.
4. After placement, FRUST snaps only the hard role atoms back to their exact role coordinates.
5. The optimizer constraints come from `constraint_spec`, not from the soft frame atoms.

Historically, `transformers.py` solved a related problem by aligning an incoming substrate onto a template substrate map with `rdMolAlign.AlignMol(...)`. That grafting-style path tied orientation to a particular template structure. The screen TS path keeps the same geometric idea but makes it more explicit: named TS roles define the reactive core, while soft frame atoms orient attached fragments during the TS guess assembly.

## Summary table

| Concept | Old transformer path | Current screen TS path |
| --- | --- | --- |
| Reactive atom identity | Fixed or positional `keep_idxs` | Named `constraint_roles` |
| Substrate orientation | `rdMolAlign.AlignMol` to template substrate atoms | Substrate soft frame anchors |
| HBpin orientation | Template-specific grafting assumptions | HBpin soft frame anchors |
| Optimizer constraints | Positional constraint atoms | `constraint_spec` role distances and angles |
| Exact snapping | Template core or coordMap behavior | Hard role atoms only |

The practical rule is: hard anchors say where the reactive atoms must be; soft frame anchors say which way the fragment should face while it is being placed.